# Jokenpô com Deep Learning em Tempo Real

Este notebook cobre toda a atividade:

1. coleta de pelo menos 45 imagens por classe pela webcam;
2. verificação e separação fixa em treino, validação e teste;
3. treinamento de uma CNN baseline;
4. curva de perda, matriz de confusão e acurácia;
5. dez rodadas em tempo real com registro automático;
6. teste adverso antes e depois de **uma única melhoria: augmentation**;
7. comparação final, capturas da webcam e arquivos para entrega.

> Execute localmente no VS Code/Jupyter. A webcam precisa estar disponível para o Python. Evite rostos e informações pessoais no enquadramento.

In [ ]:
# Instala automaticamente apenas o que estiver ausente.
import importlib.util
import subprocess
import sys

DEPENDENCIAS = {
    'cv2': 'opencv-python',
    'matplotlib': 'matplotlib',
    'numpy': 'numpy',
    'pandas': 'pandas',
    'PIL': 'pillow',
    'seaborn': 'seaborn',
    'sklearn': 'scikit-learn',
    'tensorflow': 'tensorflow',
}
ausentes = [pacote for modulo, pacote in DEPENDENCIAS.items()
            if importlib.util.find_spec(modulo) is None]
if ausentes:
    print('Instalando:', ', '.join(ausentes))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *ausentes])
    print('Instalação concluída. Se algum import falhar, reinicie o kernel e execute novamente.')

import hashlib
import random
import time
import warnings
from collections import Counter, deque
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

SEMENTE = 42
random.seed(SEMENTE)
np.random.seed(SEMENTE)
tf.keras.utils.set_random_seed(SEMENTE)

CLASSES = ['PEDRA', 'PAPEL', 'TESOURA']
CLASSE_PARA_ID = {classe: indice for indice, classe in enumerate(CLASSES)}
ID_PARA_CLASSE = {indice: classe for classe, indice in CLASSE_PARA_ID.items()}
PASTA_DATASET = Path('dataset_jokenpo')
PASTA_RESULTADOS = Path('resultados_jokenpo')
PASTA_MODELOS = Path('modelos_jokenpo')
EXTENSOES = {'.png', '.jpg', '.jpeg', '.bmp', '.webp'}
TAMANHO_IMAGEM = (128, 128)
BATCH_SIZE = 16
EPOCAS = 20

for classe in CLASSES:
    (PASTA_DATASET / classe).mkdir(parents=True, exist_ok=True)
PASTA_RESULTADOS.mkdir(exist_ok=True)
PASTA_MODELOS.mkdir(exist_ok=True)
sns.set_theme(style='whitegrid')
print('TensorFlow:', tf.__version__)
print('Pastas criadas em:', PASTA_DATASET.resolve())

## Etapa 1 — Coletar o dataset

Controles da janela da webcam:

- `1`: selecionar PEDRA;
- `2`: selecionar PAPEL;
- `3`: selecionar TESOURA;
- `S`: salvar uma imagem;
- `A`: ligar/desligar captura automática (uma imagem a cada 0,7 s);
- `Q`: encerrar.

Mantenha somente a mão dentro do retângulo verde. Varie posição, distância, ângulo, iluminação e fundo. Na captura automática, mova a mão entre as imagens para evitar duplicatas.

In [ ]:
def contagens_dataset():
    return {
        classe: sum(1 for p in (PASTA_DATASET / classe).glob('*')
                    if p.is_file() and p.suffix.lower() in EXTENSOES)
        for classe in CLASSES
    }


def limites_roi(frame):
    altura, largura = frame.shape[:2]
    lado = int(min(altura * 0.72, largura * 0.48))
    x2 = largura - 25
    x1 = max(0, x2 - lado)
    y1 = max(0, (altura - lado) // 2)
    y2 = min(altura, y1 + lado)
    return x1, y1, x2, y2


def abrir_camera(indice=0):
    camera = cv2.VideoCapture(indice)
    if not camera.isOpened():
        raise RuntimeError(
            'Não foi possível abrir a webcam. Feche outros aplicativos que estejam usando-a '
            'e confira a permissão de câmera do VS Code/Python.'
        )
    return camera


def coletar_imagens(indice_camera=0, minimo_por_classe=45):
    camera = abrir_camera(indice_camera)
    classe_atual = 'PEDRA'
    automatico = False
    ultima_captura = 0.0
    intervalo = 0.7
    janela = 'Coleta Jokenpo'

    print('A janela da webcam foi aberta. Use 1/2/3, S, A e Q.')
    try:
        while True:
            ok, frame = camera.read()
            if not ok:
                raise RuntimeError('Falha ao ler um quadro da webcam.')
            frame = cv2.flip(frame, 1)
            x1, y1, x2, y2 = limites_roi(frame)
            roi = frame[y1:y2, x1:x2]
            contagens = contagens_dataset()

            agora = time.time()
            deve_salvar = automatico and agora - ultima_captura >= intervalo
            if deve_salvar:
                destino = PASTA_DATASET / classe_atual
                nome = destino / f'{classe_atual.lower()}_{time.time_ns()}.jpg'
                cv2.imwrite(str(nome), roi)
                ultima_captura = agora

            exibicao = frame.copy()
            cv2.rectangle(exibicao, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(exibicao, f'Classe: {classe_atual}', (20, 35),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
            cv2.putText(exibicao,
                        f"PEDRA {contagens['PEDRA']} | PAPEL {contagens['PAPEL']} | TESOURA {contagens['TESOURA']}",
                        (20, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.62, (255, 255, 255), 2)
            modo = 'AUTO LIGADO' if automatico else 'MANUAL'
            cv2.putText(exibicao, f'{modo} | 1/2/3 classe | S salvar | A auto | Q sair',
                        (20, exibicao.shape[0] - 20), cv2.FONT_HERSHEY_SIMPLEX,
                        0.52, (255, 255, 255), 1)
            cv2.imshow(janela, exibicao)

            tecla = cv2.waitKey(1) & 0xFF
            if tecla == ord('1'):
                classe_atual = 'PEDRA'
            elif tecla == ord('2'):
                classe_atual = 'PAPEL'
            elif tecla == ord('3'):
                classe_atual = 'TESOURA'
            elif tecla in (ord('s'), ord('S')):
                destino = PASTA_DATASET / classe_atual
                nome = destino / f'{classe_atual.lower()}_{time.time_ns()}.jpg'
                cv2.imwrite(str(nome), roi)
                ultima_captura = time.time()
            elif tecla in (ord('a'), ord('A')):
                automatico = not automatico
                ultima_captura = time.time()
            elif tecla in (ord('q'), ord('Q')):
                break
    finally:
        camera.release()
        cv2.destroyAllWindows()

    contagens = contagens_dataset()
    print('Contagens finais:', contagens)
    faltantes = {c: minimo_por_classe - n for c, n in contagens.items() if n < minimo_por_classe}
    if faltantes:
        print('Ainda faltam imagens:', faltantes)


contagens_iniciais = contagens_dataset()
if min(contagens_iniciais.values(), default=0) < 45:
    coletar_imagens()
else:
    print('Coleta já completa:', contagens_iniciais)

## Verificação do dataset

A próxima célula verifica arquivos corrompidos, quantidade por classe e duplicatas exatas. Se houver erro, corrija a pasta e execute novamente.

In [ ]:
def verificar_dataset(minimo=45):
    registros, corrompidos = [], []
    for classe in CLASSES:
        for caminho in sorted((PASTA_DATASET / classe).glob('*')):
            if not caminho.is_file() or caminho.suffix.lower() not in EXTENSOES:
                continue
            try:
                with Image.open(caminho) as imagem:
                    imagem.verify()
                resumo = hashlib.sha256(caminho.read_bytes()).hexdigest()
                registros.append({'arquivo': str(caminho), 'classe': classe, 'hash': resumo})
            except Exception as exc:
                corrompidos.append({'arquivo': str(caminho), 'erro': str(exc)})

    df = pd.DataFrame(registros)
    if corrompidos:
        display(pd.DataFrame(corrompidos))
        raise ValueError('Remova ou substitua os arquivos corrompidos acima.')
    if df.empty:
        raise ValueError('O dataset está vazio.')

    contagens = df['classe'].value_counts().reindex(CLASSES, fill_value=0)
    display(contagens.rename('quantidade').to_frame().T)
    if (contagens < minimo).any():
        faltantes = {c: int(minimo - n) for c, n in contagens.items() if n < minimo}
        raise ValueError(f'Complete a coleta. Imagens faltantes: {faltantes}')

    duplicadas = df[df.duplicated('hash', keep=False)].sort_values('hash')
    if len(duplicadas):
        warnings.warn(f'Foram encontradas {len(duplicadas)} ocorrências de arquivos idênticos.')
        display(duplicadas[['arquivo', 'classe']])
    return df


df_arquivos = verificar_dataset()

amostra = (df_arquivos.groupby('classe', group_keys=False)
           .sample(n=min(5, df_arquivos.groupby('classe').size().min()), random_state=SEMENTE))
fig, eixos = plt.subplots(3, 5, figsize=(12, 7))
for eixo, (_, linha) in zip(eixos.flat, amostra.iterrows()):
    with Image.open(linha['arquivo']) as imagem:
        eixo.imshow(imagem.convert('RGB'))
    eixo.set_title(linha['classe'])
    eixo.axis('off')
plt.suptitle('Amostra do dataset coletado')
plt.tight_layout()
plt.show()

## Etapa 2 — Separar treino, validação e teste

O teste fica reservado e será exatamente o mesmo para a CNN baseline e para a CNN com augmentation. Isso torna a comparação controlada.

In [ ]:
caminhos = df_arquivos['arquivo'].to_numpy()
rotulos = df_arquivos['classe'].map(CLASSE_PARA_ID).to_numpy()

caminhos_treino_val, caminhos_teste, y_treino_val, y_teste = train_test_split(
    caminhos, rotulos, test_size=0.20, random_state=SEMENTE, stratify=rotulos
)
caminhos_treino, caminhos_val, y_treino, y_val = train_test_split(
    caminhos_treino_val, y_treino_val, test_size=0.20,
    random_state=SEMENTE, stratify=y_treino_val
)

def ler_imagem(caminho, rotulo):
    conteudo = tf.io.read_file(caminho)
    imagem = tf.io.decode_image(conteudo, channels=3, expand_animations=False)
    imagem.set_shape([None, None, 3])
    imagem = tf.image.resize(imagem, TAMANHO_IMAGEM)
    return imagem, rotulo

def criar_dataset(caminhos_ds, rotulos_ds, treino=False):
    ds = tf.data.Dataset.from_tensor_slices((caminhos_ds, rotulos_ds))
    ds = ds.map(ler_imagem, num_parallel_calls=tf.data.AUTOTUNE)
    if treino:
        ds = ds.shuffle(len(caminhos_ds), seed=SEMENTE, reshuffle_each_iteration=False)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

ds_treino = criar_dataset(caminhos_treino, y_treino, treino=True)
ds_val = criar_dataset(caminhos_val, y_val)
ds_teste = criar_dataset(caminhos_teste, y_teste)

resumo_divisao = pd.DataFrame({
    'divisão': ['treino', 'validação', 'teste'],
    'imagens': [len(y_treino), len(y_val), len(y_teste)],
})
display(resumo_divisao)

## CNN baseline — sem augmentation

A arquitetura é pequena para funcionar com um dataset de aproximadamente 135 imagens. O modelo recebe apenas o recorte da região verde da webcam.

In [ ]:
def criar_cnn(usar_augmentation=False):
    camadas = [tf.keras.layers.Input(shape=(*TAMANHO_IMAGEM, 3))]
    if usar_augmentation:
        camadas.extend([
            tf.keras.layers.RandomFlip('horizontal', seed=SEMENTE),
            tf.keras.layers.RandomRotation(0.10, seed=SEMENTE),
            tf.keras.layers.RandomTranslation(0.08, 0.08, seed=SEMENTE),
            tf.keras.layers.RandomZoom(0.10, seed=SEMENTE),
        ])
    camadas.extend([
        tf.keras.layers.Rescaling(1.0 / 255),
        tf.keras.layers.Conv2D(16, 3, padding='same', activation='relu'),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Dropout(0.30),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(len(CLASSES), activation='softmax'),
    ])
    modelo = tf.keras.Sequential(camadas)
    modelo.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return modelo

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEMENTE)
modelo_base = criar_cnn(usar_augmentation=False)
pesos_iniciais = [peso.copy() for peso in modelo_base.get_weights()]
modelo_base.summary()
historico_base = modelo_base.fit(
    ds_treino, validation_data=ds_val, epochs=EPOCAS, verbose=1
)
modelo_base.save(PASTA_MODELOS / 'cnn_jokenpo_baseline.keras')

In [ ]:
def plotar_historico(historico, titulo, nome_arquivo):
    epocas = range(1, len(historico.history['loss']) + 1)
    fig, eixos = plt.subplots(1, 2, figsize=(12, 4))
    eixos[0].plot(epocas, historico.history['loss'], label='treino')
    eixos[0].plot(epocas, historico.history['val_loss'], label='validação')
    eixos[0].set(title=f'Perda — {titulo}', xlabel='Época', ylabel='Loss')
    eixos[0].legend()
    eixos[1].plot(epocas, historico.history['accuracy'], label='treino')
    eixos[1].plot(epocas, historico.history['val_accuracy'], label='validação')
    eixos[1].set(title=f'Acurácia — {titulo}', xlabel='Época', ylabel='Acurácia', ylim=(0, 1.05))
    eixos[1].legend()
    plt.tight_layout()
    fig.savefig(PASTA_RESULTADOS / nome_arquivo, dpi=160, bbox_inches='tight')
    plt.show()


def avaliar_modelo(modelo, nome, nome_arquivo):
    probabilidades = modelo.predict(ds_teste, verbose=0)
    previstos = probabilidades.argmax(axis=1)
    acuracia = accuracy_score(y_teste, previstos)
    matriz = confusion_matrix(y_teste, previstos, labels=range(len(CLASSES)))
    plt.figure(figsize=(6, 5))
    sns.heatmap(matriz, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASSES, yticklabels=CLASSES)
    plt.title(f'Matriz de confusão — {nome} | acurácia: {acuracia:.1%}')
    plt.xlabel('Previsto')
    plt.ylabel('Real')
    plt.tight_layout()
    plt.savefig(PASTA_RESULTADOS / nome_arquivo, dpi=160, bbox_inches='tight')
    plt.show()
    print(classification_report(y_teste, previstos, target_names=CLASSES, zero_division=0))
    return acuracia, previstos


plotar_historico(historico_base, 'baseline', 'curvas_baseline.png')
acuracia_base, previstos_base = avaliar_modelo(
    modelo_base, 'baseline', 'matriz_confusao_baseline.png'
)
print(f'Acurácia baseline no teste: {acuracia_base:.2%}')

## Etapa 3 — Dez rodadas contra a CNN

Na janela da webcam:

- pressione `1`, `2` ou `3` para informar o gesto **real** mostrado;
- pressione `R` para registrar a rodada;
- pressione `C` para salvar uma captura adicional;
- pressione `Q` para encerrar antes das dez rodadas.

A primeira rodada já salva automaticamente uma captura com a previsão. A jogada do computador é aleatória; o modelo é responsável por reconhecer o seu gesto.

In [ ]:
def resultado_jokenpo(jogador, computador):
    if jogador == computador:
        return 'EMPATE'
    vence = {('PEDRA', 'TESOURA'), ('TESOURA', 'PAPEL'), ('PAPEL', 'PEDRA')}
    return 'JOGADOR VENCEU' if (jogador, computador) in vence else 'COMPUTADOR VENCEU'


def prever_roi(modelo, roi):
    rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
    imagem = cv2.resize(rgb, TAMANHO_IMAGEM).astype(np.float32)
    return modelo.predict(imagem[None, ...], verbose=0)[0]


def executar_teste_webcam(modelo, nome_teste, condicao, n_rodadas=10, indice_camera=0):
    camera = abrir_camera(indice_camera)
    historico_prob = deque(maxlen=5)
    gesto_real = None
    registros = []
    rng = random.Random(SEMENTE + sum(ord(c) for c in nome_teste))
    captura_salva = False
    janela = f'Jokenpo - {nome_teste}'

    print(f'Teste: {nome_teste} | condição: {condicao}')
    print('Selecione o gesto real com 1/2/3 e pressione R para registrar.')
    try:
        while len(registros) < n_rodadas:
            ok, frame = camera.read()
            if not ok:
                raise RuntimeError('Falha ao ler um quadro da webcam.')
            frame = cv2.flip(frame, 1)
            x1, y1, x2, y2 = limites_roi(frame)
            roi = frame[y1:y2, x1:x2]
            historico_prob.append(prever_roi(modelo, roi))
            probs = np.mean(historico_prob, axis=0)
            indice = int(np.argmax(probs))
            previsto = ID_PARA_CLASSE[indice]
            confianca = float(probs[indice])

            exibicao = frame.copy()
            cv2.rectangle(exibicao, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(exibicao, f'Previsto: {previsto} ({confianca:.1%})',
                        (20, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 255, 255), 2)
            real_txt = gesto_real if gesto_real else 'selecione 1/2/3'
            cv2.putText(exibicao, f'Real: {real_txt} | Rodada {len(registros) + 1}/{n_rodadas}',
                        (20, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.62, (255, 255, 255), 2)
            cv2.putText(exibicao, f'Condicao: {condicao}', (20, 102),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.58, (255, 255, 255), 1)
            cv2.putText(exibicao, '1 PEDRA | 2 PAPEL | 3 TESOURA | R registrar | Q sair',
                        (20, exibicao.shape[0] - 20), cv2.FONT_HERSHEY_SIMPLEX,
                        0.48, (255, 255, 255), 1)
            cv2.imshow(janela, exibicao)

            tecla = cv2.waitKey(1) & 0xFF
            if tecla == ord('1'):
                gesto_real = 'PEDRA'
            elif tecla == ord('2'):
                gesto_real = 'PAPEL'
            elif tecla == ord('3'):
                gesto_real = 'TESOURA'
            elif tecla in (ord('c'), ord('C')):
                cv2.imwrite(str(PASTA_RESULTADOS / f'captura_{nome_teste}_{time.time_ns()}.png'), exibicao)
            elif tecla in (ord('r'), ord('R')):
                if gesto_real is None:
                    print('Escolha primeiro o gesto real com 1, 2 ou 3.')
                    continue
                jogada_computador = rng.choice(CLASSES)
                registros.append({
                    'rodada': len(registros) + 1,
                    'gesto_real': gesto_real,
                    'gesto_previsto': previsto,
                    'confianca': confianca,
                    'acerto': previsto == gesto_real,
                    'jogada_computador': jogada_computador,
                    'resultado_jogo': resultado_jokenpo(gesto_real, jogada_computador),
                    'condicao': condicao,
                    'modelo': nome_teste,
                })
                if not captura_salva:
                    cv2.imwrite(str(PASTA_RESULTADOS / f'captura_{nome_teste}.png'), exibicao)
                    captura_salva = True
                print(registros[-1])
                gesto_real = None
                historico_prob.clear()
            elif tecla in (ord('q'), ord('Q')):
                break
    finally:
        camera.release()
        cv2.destroyAllWindows()

    df = pd.DataFrame(registros)
    destino = PASTA_RESULTADOS / f'{nome_teste}.csv'
    df.to_csv(destino, index=False)
    print(f'Registro salvo em {destino}. Rodadas: {len(df)}')
    if len(df):
        display(df)
        print(f'Acurácia nas rodadas: {df["acerto"].mean():.1%}')
    return df


rodadas_normais = executar_teste_webcam(
    modelo_base, '10_rodadas_normal_baseline', 'normal', n_rodadas=10
)

## Etapa 5 — Uma melhoria controlada: augmentation

A única mudança será ativar transformações aleatórias nas imagens durante o treinamento. A divisão dos dados, arquitetura, pesos iniciais, ordem dos lotes, épocas, batch size e taxa de aprendizado permanecem iguais.

In [ ]:
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEMENTE)
modelo_melhorado = criar_cnn(usar_augmentation=True)
modelo_melhorado.set_weights(pesos_iniciais)
historico_melhorado = modelo_melhorado.fit(
    ds_treino, validation_data=ds_val, epochs=EPOCAS, verbose=1
)
modelo_melhorado.save(PASTA_MODELOS / 'cnn_jokenpo_augmentation.keras')

plotar_historico(historico_melhorado, 'com augmentation', 'curvas_augmentation.png')
acuracia_melhorada, previstos_melhorados = avaliar_modelo(
    modelo_melhorado, 'com augmentation', 'matriz_confusao_augmentation.png'
)

comparacao_teste = pd.DataFrame({
    'modelo': ['baseline', 'augmentation'],
    'acurácia_teste': [acuracia_base, acuracia_melhorada],
})
comparacao_teste['diferença_para_baseline'] = (
    comparacao_teste['acurácia_teste'] - acuracia_base
)
display(comparacao_teste)
comparacao_teste.to_csv(PASTA_RESULTADOS / 'comparacao_teste.csv', index=False)

## Checklist da entrega

Ao terminar, entregue:

- `dataset_jokenpo/PEDRA`, `PAPEL` e `TESOURA`;
- este notebook executado e salvo;
- `resultados_jokenpo/curvas_baseline.png` e `curvas_augmentation.png`;
- matrizes de confusão em `resultados_jokenpo/`;
- CSV das dez rodadas normais;
- CSVs e comparação do teste adverso;
- pelo menos uma `captura_*.png`;
- `conclusao.txt`, contendo hipótese, melhoria, ganho/queda, efeito colateral e limitação.


### Referências técnicas

- [TensorFlow — classificação de imagens e augmentation](https://www.tensorflow.org/tutorials/images/classification)
- [OpenCV — VideoCapture](https://docs.opencv.org/4.x/d8/dfe/classcv_1_1VideoCapture.html)